# Import Photometer Metadata to DB

In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
import requests
import os

#### Definition DB

In [2]:
# Define the database path
db_name = '../data/TessNetwork_metadata.db'

# Check if the database file exists
if not os.path.isfile(db_name):
    print("Database does not exist. Creating a new one.")
else:
    print("Database already exists.")

# Create the engine regardless, since it will just connect to the existing database
engine = create_engine(f'sqlite:///{db_name}')

# SQL query to drop the table if it exists
drop_table_query = "DROP TABLE IF EXISTS TessNetwork_metadata;"

# SQL query to create the table
create_table_query = '''
CREATE TABLE TessNetwork_metadata (
    name VARCHAR(255),
    latitude DECIMAL(10, 2),
    longitude DECIMAL(10, 2),
    country VARCHAR(255),
    city VARCHAR(255),
    place VARCHAR(255),
    local_timezone_original VARCHAR(50),
    local_timezone_mapping VARCHAR(50),
    org_name VARCHAR(255),
    org_web_url VARCHAR(255)
);
'''

# Connect to the database, drop the table, and then create a new one
with engine.connect() as connection:
    # Drop the existing table if it exists
    connection.execute(text(drop_table_query))
    print("Table dropped successfully (if it existed).")
    
    # Create the table
    connection.execute(text(create_table_query))
    print("Table created successfully.")

Database already exists.
Table dropped successfully (if it existed).
Table created successfully.


#### API Metadata to SQLite Database

In [6]:
# API URL
api_url = "https://api.stars4all.eu/photometers"

# Fetch data from the API
response = requests.get(api_url)
data = response.json()

# Parse data and load it into a list of dictionaries
records = []
for item in data:
    record = {
        "name": item.get("name"),
        "latitude": round(float(item.get("latitude")), 2) if item.get("latitude") is not None else None,
        "longitude": round(float(item.get("longitude")), 2) if item.get("longitude") is not None else None,
        "country": item.get("country", item.get("info_location", {}).get("country")),
        "city": item.get("city", item.get("info_location", {}).get("town")),
        "place": item.get("place", item.get("info_location", {}).get("place")),
        "local_timezone_original": item.get("local_timezone", item.get("info_tess", {}).get("local_timezone")),
        "local_timezone_mapping": '',  # Default value
        "org_name": item.get("info_org", {}).get("name"),
        "org_web_url": item.get("info_org", {}).get("web_url")
    }
    records.append(record)

# Convert to DataFrame
df = pd.DataFrame(records)

# Insert new data into SQLite database
df.to_sql('TessNetwork_metadata', con=engine, if_exists='append', index=False)

print("New data inserted successfully!")

New data inserted successfully!


In [24]:
# Read the table from the Wikipedia page
df_tz = pd.read_html("https://en.wikipedia.org/wiki/List_of_tz_database_time_zones")

In [31]:

# Select the first table on the page
time_zone_table = df_tz[0]

# Extract the specific columns
timezone_mapping = time_zone_table.loc[:, [('TZ identifier', 'TZ identifier'), ('UTC offset ±hh:mm', 'SDT')]]

# Rename columns for simplicity
timezone_mapping.columns = ['TZ_identifier', 'UTC_offset_hh_mm']

# Display the result
print(timezone_mapping)






          TZ_identifier UTC_offset_hh_mm
0        Africa/Abidjan           +00:00
1          Africa/Accra           +00:00
2    Africa/Addis_Ababa           +03:00
3        Africa/Algiers           +01:00
4         Africa/Asmara           +03:00
..                  ...              ...
592            US/Samoa           −11:00
593                 UTC           +00:00
594                W-SU           +03:00
595                 WET           +00:00
596                Zulu           +00:00

[597 rows x 2 columns]


In [33]:
import requests
import pandas as pd

# API URL
api_url = "https://api.stars4all.eu/photometers"

# Fetch data from the API
response = requests.get(api_url)
data = response.json()

# Parse data and load it into a list of dictionaries
records = []
for item in data:
    record = {
        "name": item.get("name"),
        "latitude": round(float(item.get("latitude")), 2) if item.get("latitude") is not None else None,
        "longitude": round(float(item.get("longitude")), 2) if item.get("longitude") is not None else None,
        "country": item.get("country", item.get("info_location", {}).get("country")),
        "city": item.get("city", item.get("info_location", {}).get("town")),
        "place": item.get("place", item.get("info_location", {}).get("place")),
        "local_timezone_original": item.get("local_timezone", item.get("info_tess", {}).get("local_timezone")),
        "local_timezone_mapping": '',  # Placeholder for the mapped value
        "org_name": item.get("info_org", {}).get("name"),
        "org_web_url": item.get("info_org", {}).get("web_url")
    }
    records.append(record)

# Convert records to DataFrame
df = pd.DataFrame(records)

# Assuming `df_tz[0]` is the DataFrame containing the timezone mapping
time_zone_table = df_tz[0]

# Extract the relevant columns from `time_zone_table` into the `timezone_mapping` DataFrame
timezone_mapping = time_zone_table.loc[:, [('TZ identifier', 'TZ identifier'), ('UTC offset ±hh:mm', 'SDT')]]

# Rename columns for simplicity
timezone_mapping.columns = ['TZ_identifier', 'UTC_offset_hh_mm']

# Merge the timezone mapping with the original data
df = pd.merge(df, timezone_mapping, left_on='local_timezone_original', right_on='TZ_identifier', how='left')

# Drop the 'TZ_identifier' column if no longer needed
df.drop(columns=['TZ_identifier'], inplace=True)

# Insert the enriched data into the SQLite database
df.to_sql('TessNetwork_metadata', con=engine, if_exists='append', index=False)

print("New data inserted successfully!")


DatabaseError: Execution failed on sql 'SELECT TZ_identifier, UTC_offset_hh_mm FROM timezones_table': no such table: timezones_table